# 2. Vector Data Ingestion

This notebook takes the clustered EMR data, generates summaries, and ingests them into the Qdrant vector database.

In [ ]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src.config import settings
from src.transform import get_model_summaries, get_site_summaries
from src.utils import get_embeddings

from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore

In [ ]:
print("1. Loading Clustered Data...")
file_path = os.path.join("..", "output", "clustered_emr.csv")
if not os.path.exists(file_path):
    raise FileNotFoundError("clustered_emr.csv not found. Run 1_clustering.ipynb first.")

df = pd.read_csv(file_path)
print(f"Loaded {len(df)} clustered records.")

In [ ]:
print("2. Creating LangChain Documents...")
docs = []

for _, row in df.iterrows():
    content = row['combined_text']
    # Remove the combined_text from metadata to avoid duplication
    metadata = {k: str(v) for k, v in row.items() if k != 'combined_text' and pd.notna(v)}
    docs.append(Document(page_content=content, metadata=metadata))

print(f"Created {len(docs)} documents from individual records.")

In [ ]:
print("3. Generating and Appending Summaries...")
model_summaries = get_model_summaries(df)
for model, text in model_summaries.items():
    docs.append(Document(page_content=text, metadata={"type": "model_summary", "machine_model": model}))

site_summaries = get_site_summaries(df)
for site, text in site_summaries.items():
    docs.append(Document(page_content=text, metadata={"type": "site_summary", "branch_site": site}))

print(f"Total documents including summaries: {len(docs)}.")

In [ ]:
print("4. Ingesting into Qdrant...")
embeddings = get_embeddings()
url = settings.qdrant_url
collection = settings.qdrant_collection

print(f"Connecting to Qdrant at {url}, collection: {collection}")

# Note: This creates a new collection or adds to an existing one
qdrant = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    url=url,
    prefer_grpc=False, # Use HTTP by default
    collection_name=collection,
    force_recreate=True # Recreate collection to ensure clean state
)
print("Ingestion complete.")

In [ ]:
print("5. Testing Similarity Search...")
query = "Apa penyebab engine overheating?"
results = qdrant.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)